# Model Training\n
\n
This notebook focuses on training machine learning models for predicting student dropout risk using the engineered features.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# Set random seed for reproducibility\n
np.random.seed(42)

## 1. Load Engineered Features\n
\n
Load the features created in the feature engineering step.

In [ ]:
# Load engineered features\n
features_df = pd.read_csv('../datasets/engineered_features.csv')
print('Features shape:', features_df.shape)
print('Features columns:', features_df.columns.tolist())

## 2. Prepare Data for Modeling\n
\n
For this sample dataset, we'll create a synthetic target variable (dropout_risk) based on some of the features.\n
In a real scenario, you would use the actual dropout_risk column from your data.

In [ ]:
# Create synthetic target variable for demonstration\n
# Higher risk if: low attendance, low marks, high fee balance, etc.\n
risk_score = (
    (1 - features_df['avg_attendance'] / 100) * 0.3 +
    (1 - features_df['avg_marks'] / 100) * 0.3 +
    np.where(features_df['total_balance'] > 0, 0.2, 0) +
    (1 - features_df['prop_paid_fees']) * 0.2
)
# Convert to binary risk (1 = high risk, 0 = low risk)\n
features_df['dropout_risk'] = (risk_score > 0.5).astype(int)

print('Target variable distribution:')
print(features_df['dropout_risk'].value_counts())
print(f'Percentage at risk: {features_df["dropout_risk"].mean()*100:.1f}%')

## 3. Feature Selection and Splitting\n
\n
Select features for modeling and split into training and test sets.

In [ ]:
# Select features for modeling (exclude non-feature columns)\n
exclude_cols = ['student_id', 'first_name', 'last_name', 'course', 'age_group']  # non-numeric or ID columns\n
feature_cols = [col for col in features_df.columns if col not in exclude_cols and col != 'dropout_risk']

X = features_df[feature_cols]
y = features_df['dropout_risk']

print(f'Number of features: {len(feature_cols)}')
print(f'Feature names: {feature_cols}')

# Split the data\n
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set size: {X_train.shape}')
print(f'Test set size: {X_test.shape}')

## 4. Feature Scaling\n
\n
Scale features to have zero mean and unit variance for better model performance.

In [ ]:
# Initialize scaler\n
scaler = StandardScaler()

# Fit on training data and transform both train and test\n
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier interpretation\n
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

## 5. Model Training\n
\n
Train multiple models and compare their performance.

In [ ]:
# Initialize models\n
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# Train and evaluate each model\n
results = {}
for name, model in models.items():
    print(f'\nTraining {name}...')
    
    # Train model\n
    model.fit(X_train_scaled, y_train)
    
    # Make predictions\n
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics\n
    accuracy = model.score(X_test_scaled, y_test)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'auc': auc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f'{name} - Accuracy: {accuracy:.3f}, AUC: {auc:.3f}')
    print(classification_report(y_test, y_pred))

## 6. Model Comparison and Selection\n
\n
Compare the trained models and select the best performing one.

In [ ]:
# Compare models\n
print('Model Comparison:')
print('='*50)
for name, result in results.items():
    print(f'{name}:')
    print(f'  Accuracy: {result["accuracy"]:.3f}')
    print(f'  AUC: {result["auc"]:.3f}')
    print()

# Select best model based on AUC
best_model_name = max(results, key=lambda x: results[x]['auc'])
best_model = results[best_model_name]['model']
print(f'Best model: {best_model_name}')

## 7. Save Model and Preprocessing Objects\n
\n
Save the trained model, scaler, and feature columns for later use in deployment.

In [ ]:
# Save the best model\n
with open('../models/risk_model_v1.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Save the scaler\n
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature columns\n
with open('../models/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)
print('Model, scaler, and feature columns saved to ../models/')

## 8. Feature Importance (for tree-based models)\n
\n
If the best model is a tree-based model, we can examine feature importance.

In [ ]:
# Check if best model has feature_importances_ attribute\n
if hasattr(best_model, 'feature_importances_'):
    # Get feature importance\n
    importance = best_model.feature_importances_
    # Create DataFrame for easy viewing\n
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    print('Feature Importance:')
    print(importance_df)
    
    # Plot feature importance
    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df.head(10), x='importance', y='feature')
    plt.title('Top 10 Feature Importances')
    plt.tight_layout()
    plt.show()
else:
    print('Best model does not have feature importance attribute.')